In [ ]:
!pip install yfinance python-docx -q
print("Done installing.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.2 MB/s eta 0:00:00
Done installing.


In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime

def fetch_last_two(ticker):
    """Return (latest_close, prior_close) for a Yahoo Finance ticker."""
    hist = yf.Ticker(ticker).history(period="5d")
    return float(hist["Close"].iloc[-1]), float(hist["Close"].iloc[-2])

print("Fetching data...")

audusd, audusd_prior = fetch_last_two("AUDUSD=X")
asx200, asx200_prior = fetch_last_two("^AXJO")
us10y_x10, us10y_x10_prior = fetch_last_two("^TNX")
us10y, us10y_prior = us10y_x10 / 10, us10y_x10_prior / 10
gold, gold_prior = fetch_last_two("GC=F")
brent, brent_prior = fetch_last_two("BZ=F")
wti, wti_prior = fetch_last_two("CL=F")

# AU 10-year yield from the RBA's published CSV (best-effort; falls back to manual entry)
try:
    rba_url = "https://www.rba.gov.au/statistics/tables/csv/f02d-hist.csv"
    df = pd.read_csv(rba_url, skiprows=10)
    target_col = next((c for c in df.columns if "10" in c and "ear" in c), None)
    au10y = float(pd.to_numeric(df[target_col], errors="coerce").dropna().iloc[-1]) / 100
except Exception as e:
    print(f"Could not auto-fetch AU 10Y yield ({e}). Enter it manually below.")
    au10y = 0.0500  # <-- EDIT THIS if the automatic fetch fails

# --- Manual entries (no reliable free data source) ---
iron_ore = 98.27          # <-- EDIT THIS weekly from tradingeconomics.com/commodity/iron-ore
rba_cash_rate = 0.0435    # <-- EDIT after each RBA meeting (rba.gov.au)
fed_funds_mid = 0.03625   # <-- EDIT after each FOMC meeting (federalreserve.gov); midpoint of the target range

print(f"AUD/USD:        {audusd:.4f}")
print(f"ASX 200:        {asx200:,.2f}")
print(f"AU 10Y yield:   {au10y:.2%}")
print(f"US 10Y yield:   {us10y:.2%}")
print(f"RBA cash rate:  {rba_cash_rate:.2%}")
print(f"Fed funds mid:  {fed_funds_mid:.2%}")
print(f"Gold:           ${gold:,.0f}/oz")
print(f"Iron ore:       ${iron_ore:,.2f}/t")
print(f"Brent crude:    ${brent:,.2f}/bbl")
print(f"WTI crude:      ${wti:,.2f}/bbl")

today_str = datetime.now().strftime("%d-%b-%Y")

Fetching data...
Could not auto-fetch AU 10Y yield (HTTP Error 404: Not Found). Enter it manually below.
AUD/USD:        0.7025
ASX 200:        8,976.80
AU 10Y yield:   5.00%
US 10Y yield:   47.45%
RBA cash rate:  4.35%
Fed funds mid:  3.62%
Gold:           $4,049/oz
Iron ore:       $98.27/t
Brent crude:    $90.12/bbl
WTI crude:      $84.67/bbl


In [17]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

FONT = "Arial"
NAVY = "1F3864"
YELLOW = "FFF2CC"
thin = Side(style="thin", color="BFBFBF")
box = Border(left=thin, right=thin, top=thin, bottom=thin)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Dashboard"
ws.sheet_view.showGridLines = False
for col, w in zip("ABCDEF", [3, 26, 14, 14, 16, 40]):
    ws.column_dimensions[col].width = w

ws.merge_cells("B2:F2")
ws["B2"] = "Global Markets Dashboard — AUD/USD, Rates, Equities & Commodities"
ws["B2"].font = Font(name=FONT, size=14, bold=True, color="FFFFFF")
ws["B2"].fill = PatternFill("solid", fgColor=NAVY)

ws.merge_cells("B3:F3")
ws["B3"] = f"Snapshot as of {today_str} — fetched live via yfinance + RBA published data"
ws["B3"].font = Font(name=FONT, size=9, italic=True, color="595959")

ws.merge_cells("B4:F4")
warning_text = "WARNING: Rows in yellow are MANUAL entries. Double-check these are current."
ws["B4"] = warning_text
ws["B4"].font = Font(name=FONT, size=9, bold=True, color="9C5700")
ws["B4"].fill = PatternFill("solid", fgColor=YELLOW)

headers = ["Metric", "Level", "Prior", "Source", "Notes"]
hdr_row = 6
for i, h in enumerate(headers):
    c = ws.cell(row=hdr_row, column=2 + i, value=h)
    c.font = Font(name=FONT, bold=True, color="FFFFFF")
    c.fill = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="center")

MANUAL_METRICS = set()
MANUAL_METRICS.add("RBA Cash Rate")
MANUAL_METRICS.add("US Fed Funds Rate (midpoint)")
MANUAL_METRICS.add("Iron Ore 62% Fe (US$/t)")

rows = []
rows.append(("AUD/USD (spot)", audusd, audusd_prior, "Yahoo Finance", "0.0000"))
rows.append(("ASX 200", asx200, asx200_prior, "Yahoo Finance", "#,##0.00"))
rows.append(("Australia 10Y Bond Yield", au10y, None, "RBA", "0.00%"))
rows.append(("US 10Y Treasury Yield", us10y, us10y_prior, "Yahoo Finance", "0.00%"))
rows.append(("RBA Cash Rate", rba_cash_rate, None, "MANUAL - verify at rba.gov.au", "0.00%"))
rows.append(("US Fed Funds Rate (midpoint)", fed_funds_mid, None, "MANUAL - verify at federalreserve.gov", "0.00%"))
rows.append(("Gold (US$/oz)", gold, gold_prior, "Yahoo Finance", "$#,##0"))
rows.append(("Iron Ore 62% Fe (US$/t)", iron_ore, None, "MANUAL - verify at tradingeconomics.com", "$#,##0.00"))
rows.append(("Brent Crude (US$/bbl)", brent, brent_prior, "Yahoo Finance", "$#,##0.00"))
rows.append(("WTI Crude (US$/bbl)", wti, wti_prior, "Yahoo Finance", "$#,##0.00"))

r = hdr_row + 1
row_index = {}
for name, level, prior, source, fmt in rows:
    row_index[name] = r
    is_manual = name in MANUAL_METRICS
    fill = None
    if is_manual:
        fill = PatternFill("solid", fgColor=YELLOW)

    display_name = name
    if is_manual:
        display_name = "WARNING - " + name

    name_cell = ws.cell(row=r, column=2, value=display_name)
    name_cell.font = Font(name=FONT, bold=is_manual)

    lc = ws.cell(row=r, column=3, value=level)
    lc.font = Font(name=FONT, bold=True)
    lc.number_format = fmt

    if prior is not None:
        pc = ws.cell(row=r, column=4, value=prior)
        pc.number_format = fmt

    sc = ws.cell(row=r, column=5, value=source)
    sc.font = Font(name=FONT, size=9, italic=True, bold=is_manual)

    for col in range(2, 6):
        cell = ws.cell(row=r, column=col)
        cell.border = box
        if fill is not None:
            cell.fill = fill
    r += 1

au10_cell = "C" + str(row_index["Australia 10Y Bond Yield"])
us10_cell = "C" + str(row_index["US 10Y Treasury Yield"])
rba_cell = "C" + str(row_index["RBA Cash Rate"])
fed_cell = "C" + str(row_index["US Fed Funds Rate (midpoint)"])

ws.cell(row=r, column=2, value="AU-US 10Y Yield Differential").font = Font(name=FONT, bold=True)
diff_formula = "=" + au10_cell + "-" + us10_cell
diff_cell = ws.cell(row=r, column=3, value=diff_formula)
diff_cell.number_format = "0.00%"
for col in range(2, 6):
    ws.cell(row=r, column=col).border = box
r += 1

ws.cell(row=r, column=2, value="Cash Rate Differential (AU-US)").font = Font(name=FONT, bold=True)
cash_diff_formula = "=" + rba_cell + "-" + fed_cell
cash_diff_cell = ws.cell(row=r, column=3, value=cash_diff_formula)
cash_diff_cell.number_format = "0.00%"
for col in range(2, 6):
    ws.cell(row=r, column=col).border = box

XLSX_PATH = "AUD_USD_Global_Markets_Dashboard.xlsx"
wb.save(XLSX_PATH)
print("Saved " + XLSX_PATH)

Saved AUD_USD_Global_Markets_Dashboard.xlsx


In [ ]:
from docx import Document
from docx.shared import Pt, RGBColor

doc = Document()

def heading(text, level=1):
    h = doc.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)
    return h

def para(text, italic=False, size=11, color=None):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.italic = italic
    if color:
        run.font.color.rgb = color
    return p

doc.add_heading("Weekly Market Update", level=0)
para("AUD/USD · Australian & US Rates · ASX 200 · Commodities", italic=True)
para(f"Week ending {today_str}")

heading("1. Summary")
diff_pct = (au10y - us10y) * 100
cash_diff_pct = (rba_cash_rate - fed_funds_mid) * 100
para(
    f"AUD/USD last traded at {audusd:.4f} against a backdrop of an AU-US 10-year yield "
    f"differential of {diff_pct:.0f} basis points (Australia {au10y:.2%} vs US {us10y:.2%}) "
    f"and a cash-rate differential of {cash_diff_pct:.0f} basis points "
    f"(RBA {rba_cash_rate:.2%} vs Fed funds midpoint {fed_funds_mid:.2%}). "
    f"The ASX 200 stood at {asx200:,.2f}."
)

heading("2. Rates")
para(
    f"Australia's 10-year government bond yield is at {au10y:.2%}, versus {us10y:.2%} in the "
    f"US. The RBA cash rate sits at {rba_cash_rate:.2%} and the Fed funds target range midpoint "
    f"is {fed_funds_mid:.2%}. [Add this week's RBA/Fed commentary and any data releases here.]"
)

heading("3. Currency: AUD/USD")
para(
    f"AUD/USD is at {audusd:.4f}, versus {audusd_prior:.4f} previously. "
    f"[Add this week's narrative on what drove the move.]"
)

heading("4. Equities: ASX 200")
para(
    f"The ASX 200 is at {asx200:,.2f}, versus {asx200_prior:,.2f} previously. "
    f"[Add sector detail and what drove the move.]"
)

heading("5. Commodities")
table = doc.add_table(rows=1, cols=3)
table.style = "Light Grid Accent 1"
hdr_cells = table.rows[0].cells
hdr_cells[0].text, hdr_cells[1].text, hdr_cells[2].text = "Commodity", "Level", "Notes"
for name, val, note in [
    ("Gold", f"${gold:,.0f}/oz", ""),
    ("Iron ore (62% Fe)", f"${iron_ore:,.2f}/t", "Manual entry — verify weekly"),
    ("Brent crude", f"${brent:,.2f}/bbl", ""),
    ("WTI crude", f"${wti:,.2f}/bbl", ""),
]:
    row_cells = table.add_row().cells
    row_cells[0].text, row_cells[1].text, row_cells[2].text = name, val, note

heading("6. Catalysts for next week")
doc.add_paragraph("[List upcoming data releases, central bank meetings, and known risk events.]", style="List Bullet")

para(
    f"Sources: Yahoo Finance, RBA, US Federal Reserve. Data as of {today_str}. "
    f"For internal analysis use, not investment advice.",
    italic=True, size=9, color=RGBColor(0x80, 0x80, 0x80)
)

WEEKLY_PATH = "Weekly_Market_Update.docx"
doc.save(WEEKLY_PATH)
print(f"Saved {WEEKLY_PATH}")

Saved Weekly_Market_Update.docx


In [ ]:
doc2 = Document()

def heading2(text, level=1):
    h = doc2.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)
    return h

def para2(text, italic=False, size=11, color=None):
    p = doc2.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.italic = italic
    if color:
        run.font.color.rgb = color
    return p

doc2.add_heading("AUD/USD Trade Thesis", level=0)
para2("Bull / Base / Bear Scenario Analysis", italic=True)
para2(f"As of {today_str} · Spot reference: AUD/USD = {audusd:.4f}")

heading2("1. Thesis Overview")
para2("[Write 2-3 sentences on the current setup and highest-probability path over 1-3 months.]")

heading2("2. Base Case")
para2("[Target range] — [reasoning]")
doc2.add_paragraph("[Catalyst 1]", style="List Bullet")
doc2.add_paragraph("[Catalyst 2]", style="List Bullet")

heading2("3. Bull Case")
para2("[Target range] — [reasoning]")
doc2.add_paragraph("[Catalyst 1]", style="List Bullet")
doc2.add_paragraph("[Catalyst 2]", style="List Bullet")
para2("Invalidated if: [condition]", italic=True)

heading2("4. Bear Case")
para2("[Target range] — [reasoning]")
doc2.add_paragraph("[Catalyst 1]", style="List Bullet")
doc2.add_paragraph("[Catalyst 2]", style="List Bullet")
para2("Invalidated if: [condition]", italic=True)

heading2("5. Key Risks to Monitor")
doc2.add_paragraph("[Risk 1]", style="List Bullet")
doc2.add_paragraph("[Risk 2]", style="List Bullet")

para2(
    f"This document is an internal analytical exercise, not investment advice. "
    f"Spot reference as of {today_str}.",
    italic=True, size=9, color=RGBColor(0x80, 0x80, 0x80)
)

THESIS_PATH = "AUDUSD_Trade_Thesis.docx"
doc2.save(THESIS_PATH)
print(f"Saved {THESIS_PATH}")

Saved AUDUSD_Trade_Thesis.docx


In [18]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(XLSX_PATH, '/content/drive/MyDrive/' + XLSX_PATH)
shutil.copy(WEEKLY_PATH, '/content/drive/MyDrive/' + WEEKLY_PATH)
shutil.copy(THESIS_PATH, '/content/drive/MyDrive/' + THESIS_PATH)
print("Saved to your Google Drive — check the main 'My Drive' folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to your Google Drive — check the main 'My Drive' folder.
